
# Projeto Semantix / EBAC
## Análise, Segmentação e Previsão de Vendas em um E-commerce Brasileiro

Dataset: Olist Brazilian E-Commerce Dataset

Objetivo: analisar vendas, segmentar clientes com K-Means, reduzir dimensionalidade com PCA e prever valor de compras com Machine Learning.


In [ ]:

!pip -q install xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


## Carregamento dos Dados

In [ ]:

customers = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')

print(customers.shape)
print(orders.shape)
print(items.shape)
print(payments.shape)


## Receita Total e Ticket Médio

In [ ]:

receita_total = payments['payment_value'].sum()
ticket_medio = payments['payment_value'].mean()

print('Receita Total:', round(receita_total,2))
print('Ticket Médio:', round(ticket_medio,2))


## PCA e K-Means

In [ ]:

base_cluster = payments.groupby('order_id')['payment_value'].sum().reset_index()

scaler = StandardScaler()
X = scaler.fit_transform(base_cluster[['payment_value']])

pca = PCA(n_components=1)
X_pca = pca.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
base_cluster['cluster'] = kmeans.fit_predict(X)

base_cluster.head()


## Regressão

In [ ]:

X = base_cluster[['cluster']]
y = base_cluster['payment_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelos = {
    'Linear': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42)
}

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    print('\n', nome)
    print('MAE:', mean_absolute_error(y_test, pred))
    print('RMSE:', np.sqrt(mean_squared_error(y_test, pred)))
    print('R2:', r2_score(y_test, pred))
